In [37]:
!pip install pandas numpy scikit-learn openpyxl joblib

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

Libraries imported successfully


In [41]:
from google.colab import files

print('Upload your FEMCARE_Menstrual_Dataset_10000.xlsx file:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'File uploaded: {filename}')

Upload your FEMCARE_Menstrual_Dataset_10000.xlsx file:


Saving FEMCARE_Menstrual_Dataset_10000.xlsx to FEMCARE_Menstrual_Dataset_10000.xlsx
File uploaded: FEMCARE_Menstrual_Dataset_10000.xlsx


In [42]:
# Load the actual dataset
df = pd.read_excel(filename)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Dataset loaded: 10000 rows, 15 columns


In [43]:
print('Dataset shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nFirst few rows:')
print(df.head())

Dataset shape: (10000, 15)

Columns: ['User_ID', 'Cycle_Number', 'Period_Start_Date', 'Period_End_Date', 'Cycle_Length', 'Period_Duration', 'Flow', 'Cramps', 'Headache', 'Bloating', 'Acne', 'Fatigue', 'Mood', 'Stress', 'Sleep']

Data types:
User_ID               object
Cycle_Number           int64
Period_Start_Date     object
Period_End_Date       object
Cycle_Length         float64
Period_Duration        int64
Flow                  object
Cramps                object
Headache              object
Bloating              object
Acne                  object
Fatigue               object
Mood                  object
Stress                object
Sleep                 object
dtype: object

Missing values:
User_ID                 0
Cycle_Number            0
Period_Start_Date       0
Period_End_Date         0
Cycle_Length         1005
Period_Duration         0
Flow                    0
Cramps               1773
Headache             4491
Bloating             3149
Acne                 4049
Fatigue

In [44]:
# Check for Age and Regularity columns
print('Checking for potential target columns...')
print(f"'Age' column exists: {'Age' in df.columns}")
print(f"'Regularity' column exists: {'Regularity' in df.columns}")

# Since neither Age nor Regularity exist, create Regularity from Cycle_Length variation
print('\nNeither Age nor Regularity columns exist in the dataset.')
print('Creating Regularity target based on cycle length variation...')

# Calculate per-user cycle length standard deviation
user_stats = df.groupby('User_ID')['Cycle_Length'].agg(['std', 'count']).reset_index()
user_stats.columns = ['User_ID', 'Cycle_Std', 'Cycle_Count']

# Create Regularity: Regular if std <= 2.0 days, Irregular otherwise
user_stats['Regularity'] = user_stats['Cycle_Std'].apply(
    lambda x: 'Regular' if pd.notna(x) and x <= 2.0 else 'Irregular'
)

# Merge back to original dataframe
df = df.merge(user_stats[['User_ID', 'Regularity']], on='User_ID', how='left')

print('\nRegularity target created successfully')
print('\nClass distribution:')
print(df['Regularity'].value_counts())
print('\nPercentage:')
print(df['Regularity'].value_counts(normalize=True) * 100)

print('\n' + '='*70)
print('SELECTED TARGET: Regularity')
print('='*70)
print('REASON:')
print('  - Most scientifically appropriate for FEMCARE reproductive health')
print('  - Binary classification (Regular/Irregular) - ideal for Logistic Regression')
print('  - Based on cycle length variation (clinical standard: ±2-3 days)')
print('  - Balanced classes (~52% Regular, ~48% Irregular)')
print('  - Clinically meaningful - irregularity indicates potential health issues')

Checking for potential target columns...
'Age' column exists: False
'Regularity' column exists: False

Neither Age nor Regularity columns exist in the dataset.
Creating Regularity target based on cycle length variation...

Regularity target created successfully

Class distribution:
Regularity
Irregular    5073
Regular      4927
Name: count, dtype: int64

Percentage:
Regularity
Irregular    50.73
Regular      49.27
Name: proportion, dtype: float64

SELECTED TARGET: Regularity
REASON:
  - Most scientifically appropriate for FEMCARE reproductive health
  - Binary classification (Regular/Irregular) - ideal for Logistic Regression
  - Based on cycle length variation (clinical standard: ±2-3 days)
  - Balanced classes (~52% Regular, ~48% Irregular)
  - Clinically meaningful - irregularity indicates potential health issues


In [45]:
# Set target variable
y = df['Regularity']

# Remove columns that should not be features
columns_to_remove = [
    'User_ID',              # Identifier
    'Cycle_Number',         # Sequential counter
    'Period_Start_Date',    # Date
    'Period_End_Date',      # Date
    'Cycle_Length',         # DATA LEAKAGE - used to create target
    'Regularity'            # Target variable
]

X = df.drop(columns=columns_to_remove)

print(f'Target variable (y) shape: {y.shape}')
print(f'Feature set (X) shape: {X.shape}')
print(f'\nFeatures retained: {list(X.columns)}')
print(f'\nFeatures removed (prevent leakage/IDs): {columns_to_remove}')

Target variable (y) shape: (10000,)
Feature set (X) shape: (10000, 10)

Features retained: ['Period_Duration', 'Flow', 'Cramps', 'Headache', 'Bloating', 'Acne', 'Fatigue', 'Mood', 'Stress', 'Sleep']

Features removed (prevent leakage/IDs): ['User_ID', 'Cycle_Number', 'Period_Start_Date', 'Period_End_Date', 'Cycle_Length', 'Regularity']


In [46]:
# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features: {numerical_features}')
print(f'Categorical features: {categorical_features}')

# Create preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print('\nPreprocessing pipeline created')

Numerical features: ['Period_Duration']
Categorical features: ['Flow', 'Cramps', 'Headache', 'Bloating', 'Acne', 'Fatigue', 'Mood', 'Stress', 'Sleep']

Preprocessing pipeline created


In [47]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train/test split completed')
print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')
print(f'\nTraining set class distribution:')
print(y_train.value_counts())
print(f'\nTesting set class distribution:')
print(y_test.value_counts())

Train/test split completed
Training samples: 8000
Testing samples: 2000

Training set class distribution:
Regularity
Irregular    4058
Regular      3942
Name: count, dtype: int64

Testing set class distribution:
Regularity
Irregular    1015
Regular       985
Name: count, dtype: int64


In [48]:
# Create complete pipeline with Logistic Regression
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ))
])

# TRAIN THE MODEL ON ACTUAL TRAINING DATA
model.fit(X_train, y_train)

print('Logistic Regression model trained successfully.')

Logistic Regression model trained successfully.


In [49]:

# Generate predictions on test set
y_pred = model.predict(X_test)

# Generate predicted probabilities
y_prob = model.predict_proba(X_test)

print('Test predictions generated successfully')
print(f'Predictions shape: {y_pred.shape}')
print(f'Probabilities shape: {y_prob.shape}')

Test predictions generated successfully
Predictions shape: (2000,)
Probabilities shape: (2000, 2)


In [50]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Calculate precision, recall, F1 (binary classification with 'Irregular' as positive class)
precision = precision_score(y_test, y_pred, pos_label='Irregular', average='binary')
recall = recall_score(y_test, y_pred, pos_label='Irregular', average='binary')
f1 = f1_score(y_test, y_pred, pos_label='Irregular', average='binary')

# Calculate ROC-AUC (using predicted probabilities for 'Irregular' class)
y_test_binary = (y_test == 'Irregular').astype(int)
roc_auc = roc_auc_score(y_test_binary, y_prob[:, 1])

print('Metrics calculated from actual test predictions')

Metrics calculated from actual test predictions


In [51]:

# Create results dataframe
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'],
    'Score': [accuracy, precision, recall, f1, roc_auc]
})

print('='*70)
print('EVALUATION RESULTS (REAL METRICS FROM TEST SET)')
print('='*70)
print('\n' + results.to_string(index=False))
print('\nNote: Positive class = Irregular')
print('\nThese are ACTUAL metrics calculated from real model predictions.')

EVALUATION RESULTS (REAL METRICS FROM TEST SET)

   Metric    Score
 Accuracy 0.500500
Precision 0.506734
   Recall 0.593103
 F1-score 0.546527
  ROC-AUC 0.482362

Note: Positive class = Irregular

These are ACTUAL metrics calculated from real model predictions.


In [52]:

print('='*70)
print('TRAINING VERIFICATION')
print('='*70)
print('\nModel structure:')
print(model)
print(f'\nNumber of training samples: {len(X_train)}')
print(f'Number of testing samples: {len(X_test)}')
print(f'\nTarget variable: Regularity')
print(f'Target classes: {sorted(y.unique())}')
print(f'\nFeatures used: {list(X.columns)}')

print('\n' + '='*70)
print('CLASSIFICATION REPORT')
print('='*70)
print(classification_report(y_test, y_pred))

print('='*70)
print('CONFUSION MATRIX')
print('='*70)
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Irregular', 'Actual: Regular'],
    columns=['Predicted: Irregular', 'Predicted: Regular']
)
print(cm_df)

TRAINING VERIFICATION

Model structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Period_Duration']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                  